# Animation Engine — L4 24 GB + Low-Disk Colab

Dedicated animation runtime: **humanoid 3D asset → Make-It-Animatable rig → ARDY motion → retarget → validated Unreal FBX/ZIP**.

Use this in a **fresh runtime separate from TRELLIS.2**. Recommended GPU: **L4 24 GB**. ARDY is officially tested on RTX 4090-class 24 GB hardware; an A100 is not required for the normal animation path.

This notebook is disk-conscious: pip cache is disabled, temporary MIA weight staging is deleted immediately, and Conda/build/apt caches are cleaned after installation.


In [ ]:
# 1. Fresh-runtime setup + GPU/disk checks
import os, pathlib, shutil, subprocess, time, collections

LOG_DIR = pathlib.Path('/content/engine_logs')
LOG_DIR.mkdir(parents=True, exist_ok=True)

def disk_status(label='disk', minimum_free_gib=None):
    total, used, free = shutil.disk_usage('/content')
    GiB = 1024**3
    print(f'[DISK] {label}: used={used/GiB:.1f} GiB | free={free/GiB:.1f} GiB | total={total/GiB:.1f} GiB', flush=True)
    if minimum_free_gib is not None and free/GiB < minimum_free_gib:
        raise RuntimeError(f'Only {free/GiB:.1f} GiB free; need at least {minimum_free_gib} GiB for the next stage.')
    return free/GiB

def run_live(cmd, *, cwd=None, env=None, label='process'):
    cmd = [str(x) for x in cmd]
    safe = ''.join(ch if ch.isalnum() or ch in '-_' else '_' for ch in label)[:60]
    log_path = LOG_DIR / f"{time.strftime('%Y%m%d_%H%M%S')}_{safe}.log"
    print('\n' + '='*78, flush=True)
    print(f'[RUN] {label}', flush=True)
    print('[CMD] ' + ' '.join(cmd), flush=True)
    print(f'[LOG] {log_path}', flush=True)
    print('='*78, flush=True)
    started = time.time(); tail = collections.deque(maxlen=100)
    with log_path.open('w', encoding='utf-8', errors='replace') as log:
        proc = subprocess.Popen(cmd, cwd=str(cwd) if cwd else None, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='', flush=True); log.write(line); log.flush(); tail.append(line.rstrip())
        rc = proc.wait()
    elapsed = time.time() - started
    if rc != 0:
        print('\n' + '!'*78, flush=True)
        print(f'[FAILED] {label} | exit={rc} | elapsed={elapsed/60:.1f} min', flush=True)
        print(f'[FULL LOG] {log_path}', flush=True)
        for line in tail: print(line, flush=True)
        print('!'*78, flush=True)
        raise RuntimeError(f'{label} failed with exit code {rc}. See {log_path}.')
    print(f'\n[DONE] {label} | elapsed={elapsed/60:.1f} min', flush=True)
    return log_path

if shutil.which('nvidia-smi') is None:
    raise RuntimeError('No NVIDIA GPU attached. In Colab choose Runtime → Change runtime type → L4 (recommended), then reconnect.')
smi = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader,nounits'], text=True, capture_output=True, check=True).stdout.strip().splitlines()
gpu_name, memory_mib = [x.strip() for x in smi[0].rsplit(',',1)]
memory_mib = int(memory_mib)
print(f'[GPU] {gpu_name} | VRAM={memory_mib/1024:.1f} GiB', flush=True)
if memory_mib < 22000:
    raise RuntimeError('This notebook targets >=22 GiB VRAM. L4 24 GB is recommended; T4 16 GB is too tight for the default ARDY Llama text encoder.')
disk_status('fresh runtime', minimum_free_gib=65)

REPO = pathlib.Path('/content/My-works')
if REPO.exists(): shutil.rmtree(REPO)
run_live(['git','clone','--progress','--depth','1','https://github.com/Logan17de/My-works.git',str(REPO)], label='Clone My-works helpers')
ENGINE_ROOT = REPO / 'ai-3d-animation-engines'
TOOLS_ANIM = ENGINE_ROOT / 'animation-engine'


## 2. Optional Google Drive build cache
Caches source snapshots/native wheels only. Large ARDY/Llama/MIA model weights stay on local Colab storage for the current session.


In [ ]:
USE_DRIVE_BUILD_CACHE = True #@param {type:'boolean'}
DRIVE_CACHE_ROOT = '/content/drive/MyDrive/AI3D_Engine_Cache' #@param {type:'string'}
if USE_DRIVE_BUILD_CACHE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    root = pathlib.Path(DRIVE_CACHE_ROOT)
    for name in ('sources','wheels','downloads'): (root/name).mkdir(parents=True, exist_ok=True)
    os.environ['ENGINE_CACHE_ROOT'] = str(root)
    print('[CACHE] Enabled:', root)
else:
    os.environ.pop('ENGINE_CACHE_ROOT', None)
    print('[CACHE] Disabled')


## 3. Hugging Face authentication + access precheck
Before running this, the same HF account must have access to **`jasongzy/Mixamo`** and **`meta-llama/Meta-Llama-3-8B-Instruct`**. Put a READ token in Colab Secrets as `HF_TOKEN`, or use the hidden prompt.


In [ ]:
import getpass, sys
from google.colab import userdata
subprocess.run([sys.executable,'-m','pip','install','-q','--no-cache-dir','huggingface_hub','hf_xet'], check=True)
from huggingface_hub import HfApi, hf_hub_download

HF_TOKEN = None
try: HF_TOKEN = userdata.get('HF_TOKEN')
except Exception: pass
if not HF_TOKEN: HF_TOKEN = getpass.getpass('Hugging Face READ token (hidden): ').strip()
if not HF_TOKEN: raise RuntimeError('HF_TOKEN is required.')
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HOME'] = '/content/huggingface'
os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'

api = HfApi(token=HF_TOKEN)
who = api.whoami(token=HF_TOKEN)
print('[HF] Logged in as:', who.get('name') or who.get('fullname') or 'authenticated user')

print('[HF] Checking gated Mixamo access...', flush=True)
mixamo_files = api.list_repo_files('jasongzy/Mixamo', repo_type='dataset', token=HF_TOKEN)
bone = next((f for f in mixamo_files if pathlib.PurePosixPath(f).name.startswith('bones') and f.endswith('.fbx')), None)
if not bone: raise RuntimeError('Mixamo access check succeeded but no bones*.fbx template was found.')
hf_hub_download('jasongzy/Mixamo', filename=bone, repo_type='dataset', token=HF_TOKEN, local_dir='/tmp/hf_access_check_mixamo')
print('[HF] ✅ Mixamo gated dataset access OK')

print('[HF] Checking Meta-Llama-3-8B-Instruct access...', flush=True)
hf_hub_download('meta-llama/Meta-Llama-3-8B-Instruct', filename='config.json', token=HF_TOKEN, local_dir='/tmp/hf_access_check_llama')
print('[HF] ✅ Llama gated model access OK')
shutil.rmtree('/tmp/hf_access_check_mixamo', ignore_errors=True)
shutil.rmtree('/tmp/hf_access_check_llama', ignore_errors=True)


## 4. Install Animation Engine — low-disk mode
This installs only ARDY + Make-It-Animatable. TRELLIS is intentionally absent from this runtime.


In [ ]:
installer = TOOLS_ANIM / 'install_animation_low_disk.sh'
run_live(['bash','-n',str(installer)], label='Animation low-disk installer syntax check')
install_env = os.environ.copy()
install_env['HF_TOKEN'] = HF_TOKEN
install_env['HF_HOME'] = '/content/huggingface'
install_env['HF_XET_HIGH_PERFORMANCE'] = '1'
run_live(['bash',str(installer)], env=install_env, label='Animation Engine low-disk installation')
disk_status('after installation cleanup', minimum_free_gib=32)


## 5. Upload the humanoid and choose motion
Upload the GLB/FBX you want to animate. If it came from the TRELLIS notebook, download it there first and upload it here.


In [ ]:
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1: raise ValueError('Upload exactly one humanoid GLB/FBX/OBJ/PLY.')
TARGET_CHARACTER = f"/content/{next(iter(uploaded))}"
if pathlib.Path(TARGET_CHARACTER).suffix.lower() not in {'.glb','.fbx','.obj','.ply'}: raise ValueError('Unsupported character format.')
PROMPT = 'A person walks forward, stops, and waves with the right hand.' #@param {type:'string'}
DURATION_SECONDS = 6.0 #@param {type:'number'}
SEED = 0 #@param {type:'integer'}
TARGET_ALREADY_RIGGED = False #@param {type:'boolean'}
MIA_NO_FINGERS = True #@param {type:'boolean'}
print('Character:', TARGET_CHARACTER)
disk_status('before ARDY/MIA runtime downloads', minimum_free_gib=28)


## 6. Run complete Animation Engine
ARDY's default local LLM2Vec/Llama encoder uses CUDA BF16. On a 24 GB L4 this is the intended first test.


In [ ]:
OUTPUT_ANIM_DIR = '/content/animation_outputs'
cmd = ['python', str(TOOLS_ANIM/'run_animation_pipeline.py'), '--character', TARGET_CHARACTER, '--prompt', PROMPT, '--duration', str(DURATION_SECONDS), '--seed', str(SEED), '--output-dir', OUTPUT_ANIM_DIR]
if TARGET_ALREADY_RIGGED: cmd.append('--already-rigged')
if MIA_NO_FINGERS: cmd.append('--no-fingers')
runtime_env = os.environ.copy()
runtime_env['HF_TOKEN'] = HF_TOKEN
runtime_env['HF_HOME'] = '/content/huggingface'
runtime_env['HF_XET_HIGH_PERFORMANCE'] = '1'
runtime_env['TEXT_ENCODER_MODE'] = 'local'
run_live(cmd, env=runtime_env, label='Complete Animation Engine pipeline')
disk_status('after animation pipeline')
MOTION_PREVIEW = f'{OUTPUT_ANIM_DIR}/motion_preview.mp4'
FINAL_FBX = f'{OUTPUT_ANIM_DIR}/character_animated.fbx'
CONTRACT_REPORT = f'{OUTPUT_ANIM_DIR}/animation_contract_report.json'
PACKAGE_ZIP = f'{OUTPUT_ANIM_DIR}/unreal_character_package.zip'
for p in (MOTION_PREVIEW, FINAL_FBX, CONTRACT_REPORT, PACKAGE_ZIP):
    if not pathlib.Path(p).is_file(): raise RuntimeError(f'Missing animation output: {p}')
print('✅ Animation complete:', FINAL_FBX)


In [ ]:
# 7. Preview and download Unreal package
from IPython.display import Video, display
display(Video(MOTION_PREVIEW, embed=True))
files.download(PACKAGE_ZIP)


## Recommended workflow

```text
A100 TRELLIS runtime → generate GLB → download GLB → end runtime
                         ↓
Fresh L4 runtime → this notebook → upload GLB → animate → download Unreal ZIP
```

This keeps the two large software/model stacks from competing for the same ~113 GB local Colab disk.
